In [ ]:
"""Session 22: YOLOv9e-seg grouped five-fold OOF on Image_Annotation_rebuilt.

Use SESSION_22_KAGGLE_RUN_THIS.ipynb on Kaggle.  The default experiment trains
one fold per T4 concurrently. All three supplied folders enter grouped OOF.
"""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile

BUILD = "SESSION22-YOLOV9E-SEG-SKIN19-V2"
MODEL = "yolov9e-seg.pt"
CLASSES = []
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
AUGMENT_PREFIX = re.compile(
    r"^(?:"
    r"Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"
    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"
    r"Resize-450_400|flip-(?:-1|0|1)|"
    r"(?:hue|saturation)-(?:200|150|100|50)|"
    r"superpixels-(?:200|100|75|50|25)"
    r")"
)


def ensure_runtime_dependencies():
    """Install only missing pinned packages when the runner is pasted in Kaggle.

    The notebook normally performs this bootstrap. Keeping it here as well
    makes a direct Papermill/notebook execution recover instead of failing
    with ModuleNotFoundError before the experiment starts.
    """
    import importlib.metadata
    required = {"ultralytics": "8.3.228", "pycocotools": "2.0.10"}
    missing = []
    for package, version in required.items():
        try:
            if importlib.metadata.version(package) != version:
                missing.append(f"{package}=={version}")
        except importlib.metadata.PackageNotFoundError:
            missing.append(f"{package}=={version}")
    if not missing:
        return
    print(f"Installing Session 22 dependencies: {missing}", flush=True)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-input",
                               "--disable-pip-version-check", *missing])
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "Could not install Session 22 dependencies. Enable Internet in Kaggle, "
            "then restart and run the notebook from its first cell."
        ) from error


def resolve_runtime_script():
    """Return a real script path for fold subprocesses in both .py and notebooks."""
    source_path = Path(globals().get("__file__", "")).resolve()
    if source_path.is_file():
        return source_path
    runtime = Path("/kaggle/working/SESSION_22_runtime.py")
    # A raw Kaggle/Papermill cell has no __file__. Preserve the cell source so
    # the two independent GPU worker processes can import the same runner. Do
    # not blindly reuse runtime: it may be left over from an older notebook run.
    source = ""
    try:
        import inspect, textwrap
        raw = inspect.getsource(sys.modules[__name__])
        # Papermill/IPython sometimes wraps the cell in an indented context,
        # causing IndentationError at line 1 when compile() sees the raw source.
        # textwrap.dedent strips common leading whitespace safely.
        source = textwrap.dedent(raw).lstrip()
    except Exception:
        pass
    if "def main" not in source:
        try:
            from IPython import get_ipython
            import textwrap as _tw
            history = get_ipython().history_manager.input_hist_raw
            raw_cell = next((cell for cell in reversed(history)
                           if "def main" in cell and "train_and_predict_worker" in cell), "")
            if raw_cell:
                source = _tw.dedent(raw_cell).lstrip()
        except Exception:
            pass
    if "def main" not in source or "train_and_predict_worker" not in source:
        # The generated notebook writes the current runtime before invoking it.
        # This fallback is accepted only when its build marker is current.
        if runtime.is_file() and f'BUILD = "{BUILD}"' in runtime.read_text(errors="replace"):
            return runtime
        raise RuntimeError(
            "Session 22 could not reconstruct its GPU-worker script, and no current "
            f"{BUILD} runtime exists. Use the regenerated "
            "SESSION_22_KAGGLE_RUN_THIS.ipynb rather than a partial source cell."
        )
    try:
        compile(source, str(runtime), "exec")
    except SyntaxError as error:
        raise RuntimeError("The notebook cell source could not be compiled as the worker runtime.") from error
    runtime.parent.mkdir(parents=True, exist_ok=True)
    temporary = runtime.with_suffix(".py.tmp")
    temporary.write_text(source)
    temporary.replace(runtime)
    print(f"Wrote notebook runtime to {runtime}", flush=True)
    return runtime


def unique_output_directory(path):
    """Avoid destroying or failing on artifacts from an earlier Kaggle run."""
    path = Path(path).resolve()
    if not path.exists():
        return path
    for index in range(1, 1000):
        candidate = path.with_name(f"{path.name}_{index:02d}")
        if not candidate.exists():
            print(f"Output directory already exists; using {candidate}", flush=True)
            return candidate
    raise RuntimeError(f"Could not allocate a unique output directory beside {path}")


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    tmp.replace(path)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def valid_layout(root):
    return all((Path(root)/split/kind).is_dir()
               for split in ("train", "valid", "test") for kind in ("images", "labels"))


def load_class_names(root):
    data_path = Path(root) / "data.yaml"
    try:
        import yaml
        data = yaml.safe_load(data_path.read_text(encoding="utf-8")) or {}
    except ModuleNotFoundError:
        import ast

        lines = data_path.read_text(encoding="utf-8").splitlines()
        names_line = next((line.split(":", 1)[1].strip()
                           for line in lines if line.lstrip().startswith("names:")), "")
        count_line = next((line.split(":", 1)[1].strip()
                           for line in lines if line.lstrip().startswith("nc:")), "")
        data = {
            "names": ast.literal_eval(names_line) if names_line else None,
            "nc": int(count_line) if count_line else None,
        }
    names = data.get("names")
    if isinstance(names, dict):
        names = [names[key] for key in sorted(names, key=lambda key: int(key))]
    if not isinstance(names, (list, tuple)) or not names:
        raise ValueError(f"{data_path} must contain a non-empty names list or mapping")
    names = [str(name) for name in names]
    declared_count = data.get("nc")
    if declared_count is not None and int(declared_count) != len(names):
        raise ValueError(
            f"{data_path} declares nc={declared_count} but contains {len(names)} names"
        )
    global CLASSES
    CLASSES = names
    print(f"Loaded {len(CLASSES)} classes from {data_path}: {CLASSES}", flush=True)
    return CLASSES


def safe_extract_archives(input_root):
    """Extract attached Image_Annotation_rebuilt ZIPs only to Kaggle working."""
    if not Path("/kaggle/working").is_dir():
        return []
    found = []
    cache = Path("/kaggle/working/session22_dataset_cache")
    for archive in sorted(Path(input_root).rglob("*.zip")):
        low = archive.name.lower()
        if "image_annotation_rebuilt" not in low:
            continue
        destination = cache / sha256(archive)[:16]
        marker = destination / ".complete"
        if not marker.is_file():
            with zipfile.ZipFile(archive) as zipped:
                bad = zipped.testzip()
                if bad:
                    raise ValueError(f"Corrupt dataset archive member: {bad}")
                for item in zipped.infolist():
                    name = Path(item.filename)
                    target = (destination / name).resolve()
                    if name.is_absolute() or ".." in name.parts or not target.is_relative_to(destination.resolve()):
                        raise ValueError(f"Unsafe archive member: {item.filename}")
                    if (item.external_attr >> 16) & 0o170000 == 0o120000:
                        raise ValueError(f"Archive symlink is not allowed: {item.filename}")
                destination.mkdir(parents=True, exist_ok=True)
                zipped.extractall(destination)
            marker.write_text("ok\n")
        found.extend(p.parent for p in destination.rglob("data.yaml") if valid_layout(p.parent))
    return found


def inferred_source_key(stem):
    return AUGMENT_PREFIX.sub("", stem.split(".rf.", 1)[0], count=1)


class DisjointSet:
    def __init__(self, n): self.parent=list(range(n)); self.rank=[0]*n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        a,b=self.find(a),self.find(b)
        if a==b: return
        if self.rank[a] < self.rank[b]: a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]: self.rank[a]+=1


def _scan_and_group(root, seed):
    try:
        from PIL import Image
    except ModuleNotFoundError:
        Image = None
    rows=[]
    for old_split in ("train","valid","test"):
        images=sorted(p for p in (root/old_split/"images").iterdir()
                      if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
        for image_path in images:
            label_path=root/old_split/"labels"/f"{image_path.stem}.txt"
            if not label_path.is_file(): raise ValueError(f"Missing label: {label_path}")
            classes=set()
            for number,line in enumerate(label_path.read_text().splitlines(),1):
                values=line.split()
                if len(values)<7 or (len(values)-1)%2: raise ValueError(f"Bad label {label_path}:{number}")
                category=int(values[0]); coords=[float(v) for v in values[1:]]
                if category not in range(len(CLASSES)) or not all(math.isfinite(v) and -1e-6<=v<=1+1e-6 for v in coords):
                    raise ValueError(f"Bad class/coordinate {label_path}:{number}")
                classes.add(category)
            if not classes:
                print(f"Skipping background image with empty label: {image_path}", flush=True)
                continue
            if Image is None:
                width = height = 0
                pixel_hash = sha256(image_path)
            else:
                with Image.open(image_path) as image:
                    rgb=image.convert("RGB"); width,height=rgb.size
                    pixel_hash=hashlib.sha256(f"{width}x{height}|".encode()+rgb.tobytes()).hexdigest()
            relative=str(image_path.relative_to(root))
            rows.append({"image_id":hashlib.sha256(relative.encode()).hexdigest()[:20],
                         "old_split":old_split,"image_file":relative,
                         "label_file":str(label_path.relative_to(root)),"width":str(width),"height":str(height),
                         "classes":"|".join(CLASSES[c] for c in sorted(classes)),
                         "class_ids":"|".join(str(c) for c in sorted(classes)),
                         "source_key":inferred_source_key(image_path.stem),"pixel_sha256":pixel_hash})
        print(f"Scanned {old_split}: {len(images)} images",flush=True)
    dsu=DisjointSet(len(rows)); seen={}
    for i,row in enumerate(rows):
        for evidence in (("source",row["source_key"]),("pixel",row["pixel_sha256"])):
            if evidence in seen: dsu.union(i,seen[evidence])
            else: seen[evidence]=i
    components=defaultdict(list)
    for i in range(len(rows)): components[dsu.find(i)].append(i)
    groups={}
    for members in components.values():
        keys=sorted({rows[i]["source_key"] for i in members})
        gid="src_"+hashlib.sha256("|".join(keys).encode()).hexdigest()[:16]
        groups[gid]=members
        for i in members: rows[i]["split_group_id"]=gid
    total=[0]*len(CLASSES); records=[]; rng=random.Random(seed)
    for gid,members in groups.items():
        counts=[0]*len(CLASSES)
        for i in members:
            for c in map(int,rows[i]["class_ids"].split("|")): counts[c]+=1
        total=[a+b for a,b in zip(total,counts)]; records.append((gid,members,counts,rng.random()))
    records.sort(key=lambda x:(-len(x[1]),x[3],x[0]))
    sizes=[0]*5; fold_classes=[[0]*len(CLASSES) for _ in range(5)]; targets=[v/5 for v in total]; assignment={}
    for pos,(gid,members,counts,_) in enumerate(records):
        candidates=[pos] if pos<5 else list(range(5)); rng.shuffle(candidates)
        def cost(f):
            return sum(((fold_classes[f][c]+counts[c])/max(targets[c],1))**2 for c in range(len(CLASSES))) + .35*((sizes[f]+len(members))/(len(rows)/5))**2
        fold=min(candidates,key=lambda f:(cost(f),sizes[f],f)); assignment[gid]=fold; sizes[fold]+=len(members)
        for c in range(len(CLASSES)): fold_classes[fold][c]+=counts[c]
    for row in rows: row["cv_fold"]=str(assignment[row["split_group_id"]]); row["split"]="development"
    for fold in range(5):
        selected=[r for r in rows if int(r["cv_fold"])==fold]
        present={name for r in selected for name in r["classes"].split("|")}
        if not selected or set(CLASSES)-present: raise ValueError(f"Fold {fold} lacks data/classes")
    return rows,groups


def find_data(explicit=None, seed=42):
    if explicit:
        candidates = [Path(explicit).expanduser().resolve()]
    else:
        candidates = []
        kaggle = Path("/kaggle/input")
        if kaggle.is_dir():
            candidates.extend(
                p for p in kaggle.rglob("*")
                if p.is_dir() and p.name.lower() in {"image_annotation_rebuilt", "skin_disease_yolo"}
                and valid_layout(p)
            )
            candidates.extend(p.parent for p in kaggle.rglob("data.yaml") if valid_layout(p.parent))
            candidates.extend(safe_extract_archives(kaggle))
        # __file__ is absent when users paste this runner directly into a
        # Kaggle/Papermill cell. Kaggle input discovery above still works; the
        # current working directory is only a local-development fallback.
        here = Path(globals().get("__file__", Path.cwd())).resolve()
        if here.is_file():
            here = here.parent
        candidates.extend([
            here / "Image_Annotation_rebuilt",
            here / "skin_disease_yolo",
            here / "dataset/Image_Annotation_rebuilt",
            here / "dataset/skin_disease_yolo",
        ])
    unique = sorted({p.resolve() for p in candidates if valid_layout(p)})
    named=[p for p in unique if p.name.lower() in {"image_annotation_rebuilt", "skin_disease_yolo"}]
    if not explicit and len(named)==1: unique=named
    if len(unique) != 1:
        raise ValueError(f"Expected one supported Session 22 dataset, found {unique}. Set DATA_ROOT explicitly.")
    root=unique[0]
    load_class_names(root)
    rows,groups=_scan_and_group(root,seed); validate_dataset(root,rows)
    return root,rows,groups


def validate_dataset(root, rows):
    if not rows:
        raise ValueError("Dataset manifest is empty")
    ids, group_partition, pixel_partition = set(), {}, {}
    expected = set(CLASSES)
    for row in rows:
        image_id = row["image_id"]
        if image_id in ids:
            raise ValueError(f"Duplicate image ID: {image_id}")
        ids.add(image_id)
        split = row["split"]
        if split != "development":
            raise ValueError(f"Invalid split: {split}")
        if row["cv_fold"] not in {"0", "1", "2", "3", "4"}:
            raise ValueError(f"Invalid CV fold for {image_id}")
        partition = f"fold_{row['cv_fold']}"
        for registry, key in ((group_partition, row["split_group_id"]),
                              (pixel_partition, row["pixel_sha256"])):
            if not key or (key in registry and registry[key] != partition):
                raise ValueError(f"Cross-partition group/pixel leakage: {key}")
            registry[key] = partition
        for column in ("image_file", "label_file"):
            path = (root / row[column]).resolve()
            if not path.is_relative_to(root.resolve()) or not path.is_file():
                raise ValueError(f"Missing or unsafe {column}: {path}")
    development = rows
    observed = []
    for fold in range(5):
        valid = [r for r in development if int(r["cv_fold"]) == fold]
        train = [r for r in development if int(r["cv_fold"]) != fold]
        if not valid or not train:
            raise ValueError(f"Empty fold {fold}")
        if {r["split_group_id"] for r in train} & {r["split_group_id"] for r in valid}:
            raise ValueError(f"Group leakage in fold {fold}")
        present = {name for r in valid for name in r["classes"].split("|")}
        if not expected.issubset(present):
            raise ValueError(f"Fold {fold} lacks classes: {expected - present}")
        observed.extend(r["image_id"] for r in valid)
    if len(observed) != len(development) or set(observed) != {r["image_id"] for r in development}:
        raise ValueError("Development images do not occur exactly once across OOF folds")


def make_fold_files(root, rows, work):
    fold_dir = work / "fold_data"
    fold_dir.mkdir(parents=True, exist_ok=True)
    development = [r for r in rows if r["split"] != "test"]
    outputs = []
    for fold in range(5):
        directory = fold_dir / f"fold_{fold}"
        directory.mkdir(parents=True, exist_ok=True)
        train = [r for r in development if int(r["cv_fold"]) != fold]
        valid = [r for r in development if int(r["cv_fold"]) == fold]
        train_txt = directory / "train.txt"
        valid_txt = directory / "valid.txt"
        train_txt.write_text("\n".join(str((root / r["image_file"]).resolve()) for r in train) + "\n")
        valid_txt.write_text("\n".join(str((root / r["image_file"]).resolve()) for r in valid) + "\n")
        yaml_path = directory / "data.yaml"
        quoted_names = ", ".join(json.dumps(name) for name in CLASSES)
        yaml_path.write_text(
            f"path: {json.dumps(str(root))}\n"
            f"train: {json.dumps(str(train_txt))}\n"
            f"val: {json.dumps(str(valid_txt))}\n"
            f"nc: {len(CLASSES)}\nnames: [{quoted_names}]\n"
        )
        outputs.append({"fold": fold, "yaml": yaml_path, "train": train, "valid": valid})
    return outputs


def write_prepared_manifest(rows, destination):
    columns = ["image_id", "old_split", "image_file", "label_file", "width", "height",
               "classes", "class_ids", "source_key", "pixel_sha256", "split_group_id", "cv_fold", "split"]
    with Path(destination).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns); writer.writeheader()
        writer.writerows({key: row[key] for key in columns} for row in rows)


def build_coco_ground_truth(root, rows, destination):
    from PIL import Image
    images, annotations, ann_id, id_map = [], [], 1, {}
    development = sorted((r for r in rows if r["split"] != "test"), key=lambda r: r["image_id"])
    for image_number, row in enumerate(development, 1):
        image_path = root / row["image_file"]
        with Image.open(image_path) as image:
            width, height = image.size
        id_map[row["image_id"]] = image_number
        images.append({"id": image_number, "file_name": row["image_file"],
                       "width": width, "height": height})
        for line in (root / row["label_file"]).read_text().splitlines():
            values = line.split()
            if len(values) < 7 or (len(values) - 1) % 2:
                continue
            category = int(values[0])
            coords = [float(v) for v in values[1:]]
            xs = [min(1.0, max(0.0, coords[i])) * width for i in range(0, len(coords), 2)]
            ys = [min(1.0, max(0.0, coords[i])) * height for i in range(1, len(coords), 2)]
            area = 0.5 * abs(sum(xs[i] * ys[(i + 1) % len(xs)] - xs[(i + 1) % len(xs)] * ys[i]
                                 for i in range(len(xs))))
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            if category not in range(len(CLASSES)) or area <= 1 or x1 <= x0 or y1 <= y0:
                continue
            polygon = [coordinate for pair in zip(xs, ys) for coordinate in pair]
            annotations.append({"id": ann_id, "image_id": image_number, "category_id": category + 1,
                                "segmentation": [polygon], "area": float(area),
                                "bbox": [x0, y0, x1 - x0, y1 - y0], "iscrowd": 0})
            ann_id += 1
    payload = {
        "info": {"description": "Session 22 original-data grouped OOF ground truth", "build": BUILD},
        "licenses": [], "images": images, "annotations": annotations,
        "categories": [{"id": i + 1, "name": name, "supercategory": "skin_disease"}
                       for i, name in enumerate(CLASSES)],
    }
    write_json(destination, payload)
    return id_map


def best_epoch_from_csv(path):
    if not Path(path).is_file():
        return None
    with Path(path).open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows:
        return None
    candidates = ["metrics/mAP50-95(M)", "metrics/mAP50(M)", "metrics/mAP50-95(B)"]
    key = next((k for k in candidates if k in rows[0]), None)
    if key is None:
        return None
    index = max(range(len(rows)), key=lambda i: float(rows[i].get(key) or -1))
    return int(float(rows[index].get("epoch", index + 1)))


def train_and_predict_worker(args):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.worker_gpu)
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    os.environ.setdefault("WANDB_DISABLED", "true")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
    ensure_runtime_dependencies()
    from ultralytics import YOLO
    import torch

    root = Path(args.data_root).resolve()
    load_class_names(root)
    work = Path(args.work_dir).resolve()
    if args.worker_hours is None or args.worker_hours <= 0:
        raise ValueError("Worker training time must be positive")
    if args.worker_gpu is None or args.worker_fold not in range(5):
        raise ValueError("Worker fold/GPU arguments are invalid")
    with (work / "grouped_fold_manifest.csv").open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    fold = args.worker_fold
    development = [r for r in rows if r["split"] != "test"]
    fold_info = {
        "fold": fold,
        "yaml": work / f"fold_data/fold_{fold}/data.yaml",
        "train": [r for r in development if int(r["cv_fold"]) != fold],
        "valid": [r for r in development if int(r["cv_fold"]) == fold],
    }
    if not fold_info["yaml"].is_file():
        raise RuntimeError(f"Parent did not prepare fold YAML: {fold_info['yaml']}")
    fold_work = work / f"fold_{fold}"
    fold_work.mkdir(parents=True, exist_ok=True)
    project = fold_work / "train"
    run_name = "fit"
    model_path = Path(args.model).resolve() if Path(args.model).is_file() else args.model
    print(f"[fold {fold}] GPU {args.worker_gpu}; train={len(fold_info['train'])}; "
          f"valid={len(fold_info['valid'])}; limit={args.worker_hours:.2f} h", flush=True)
    model = YOLO(str(model_path))
    started = time.time()
    model.train(
        data=str(fold_info["yaml"]), epochs=args.epochs, time=args.worker_hours,
        patience=args.patience, imgsz=args.imgsz, batch=args.batch, device=0,
        workers=args.workers, optimizer="AdamW", lr0=args.lr0, lrf=args.lrf,
        weight_decay=args.weight_decay, warmup_epochs=3.0, cos_lr=True,
        amp=True, deterministic=True, seed=args.seed,
        # This dataset already contains extensive offline transforms; keep
        # online augmentation mild to avoid compounding those artifacts.
        mosaic=0.0, close_mosaic=0, mixup=0.0, copy_paste=0.0,
        degrees=5.0, translate=0.05, scale=0.15, shear=0.0,
        fliplr=0.5, flipud=0.1, hsv_h=0.005, hsv_s=0.20, hsv_v=0.15,
        project=str(project), name=run_name, exist_ok=True, plots=True,
        # Kaggle inputs are read-only, so do not ask Ultralytics to write image
        # cache files beside the attached dataset.
        save=True, save_period=-1, cache=False, verbose=True,
    )
    run_dir = project / run_name
    best = run_dir / "weights/best.pt"
    if not best.is_file():
        raise RuntimeError(f"Fold {fold} did not produce best.pt")

    del model
    torch.cuda.empty_cache()
    model = YOLO(str(best))
    row_by_path = {str((root / row["image_file"]).resolve()): row for row in fold_info["valid"]}
    valid_list = work / f"fold_data/fold_{fold}/valid.txt"
    if not valid_list.is_file():
        raise RuntimeError(f"Missing validation path list for fold {fold}: {valid_list}")
    predictions = []
    seen = []
    # In Ultralytics 8.3.228 a Python list of paths is converted to one large
    # in-memory PIL batch, so predict(batch=...) may not cap GPU memory. A .txt
    # source uses LoadImagesAndVideos and honors the requested inference batch.
    results = model.predict(source=str(valid_list), imgsz=args.imgsz, conf=args.conf,
                            iou=args.nms_iou, max_det=args.max_det, device=0,
                            batch=args.predict_batch, stream=True, verbose=False)
    for result in results:
        resolved = str(Path(result.path).resolve())
        row = row_by_path.get(resolved)
        if row is None:
            raise RuntimeError(f"Unexpected prediction path in fold {fold}: {result.path}")
        seen.append(row["image_id"])
        if result.boxes is None or result.masks is None:
            continue
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        scores = result.boxes.conf.detach().cpu().numpy()
        classes = result.boxes.cls.detach().cpu().numpy().astype(int)
        polygons = result.masks.xy
        for box, score, category, polygon in zip(boxes, scores, classes, polygons):
            if category not in range(len(CLASSES)) or polygon is None or len(polygon) < 3:
                continue
            flat = [float(value) for point in polygon for value in point]
            x0, y0, x1, y1 = [float(v) for v in box]
            predictions.append({"image_id_string": row["image_id"],
                                "category_id": int(category) + 1,
                                "score": float(score), "segmentation": [flat],
                                "bbox": [x0, y0, max(0.0, x1 - x0), max(0.0, y1 - y0)]})
    if len(seen) != len(fold_info["valid"]) or set(seen) != {r["image_id"] for r in fold_info["valid"]}:
        raise RuntimeError(f"Fold {fold} OOF inference is incomplete")
    pred_path = fold_work / "oof_predictions.json"
    write_json(pred_path, predictions)
    result = {
        "fold": fold, "status": "complete", "gpu": args.worker_gpu,
        "train_images": len(fold_info["train"]), "valid_images": len(fold_info["valid"]),
        "prediction_images": len(seen), "detections": len(predictions),
        "elapsed_hours": (time.time() - started) / 3600,
        "best_epoch": best_epoch_from_csv(run_dir / "results.csv"),
        "best_checkpoint": str(best), "best_checkpoint_sha256": sha256(best),
        "original_test_folder_included_in_oof": True,
    }
    write_json(fold_work / "result.json", result)
    print(f"[fold {fold}] complete; detections={len(predictions)}", flush=True)
    return 0


def coco_evaluate(gt_path, predictions, destination):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import numpy as np

    write_json(destination, predictions)
    if not predictions:
        return {task: {"AP50_95": 0.0, "AP50": 0.0, "AP75": 0.0,
                       "per_class_AP50": {name: 0.0 for name in CLASSES}}
                for task in ("bbox", "segm")}
    coco_gt = COCO(str(gt_path))
    coco_dt = coco_gt.loadRes(str(destination))
    output = {}
    for task in ("bbox", "segm"):
        evaluator = COCOeval(coco_gt, coco_dt, task)
        evaluator.params.maxDets = [1, 10, 300]
        evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
        precision = evaluator.eval["precision"]  # IoU, recall, class, area, maxDet
        iou_index = int(np.argmin(np.abs(evaluator.params.iouThrs - 0.5)))
        per_class = {}
        for index, name in enumerate(CLASSES):
            values = precision[iou_index, :, index, 0, -1]
            values = values[values > -1]
            per_class[name] = float(values.mean()) if len(values) else 0.0
        output[task] = {"AP50_95": float(evaluator.stats[0]),
                        "AP50": float(evaluator.stats[1]),
                        "AP75": float(evaluator.stats[2]),
                        "per_class_AP50": per_class}
    return output


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description="Session 22 YOLOv9e-seg grouped five-fold OOF")
    parser.add_argument("--data-root", default=None)
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--model", default=MODEL)
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--patience", type=int, default=12)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=2, help="Conservative per-fold, per-T4 batch")
    parser.add_argument("--predict-batch", type=int, default=4)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--lr0", type=float, default=0.001)
    parser.add_argument("--lrf", type=float, default=0.01)
    parser.add_argument("--weight-decay", type=float, default=0.0005)
    parser.add_argument("--conf", type=float, default=0.001)
    parser.add_argument("--nms-iou", type=float, default=0.7)
    parser.add_argument("--max-det", type=int, default=300)
    parser.add_argument("--session-hours", type=float, default=11.0)
    parser.add_argument("--reserve-minutes", type=float, default=45.0)
    parser.add_argument("--max-fold-hours", type=float, default=3.0)
    parser.add_argument("--audit-only", action="store_true")
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--worker-fold", type=int, default=None, help=argparse.SUPPRESS)
    parser.add_argument("--worker-gpu", type=int, default=None, help=argparse.SUPPRESS)
    parser.add_argument("--worker-hours", type=float, default=None, help=argparse.SUPPRESS)
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f"Ignoring notebook arguments: {unknown}")
    return args


def main(argv=None):
    args = parse_args(argv)
    if args.worker_fold is not None:
        return train_and_predict_worker(args)
    if not args.audit_only:
        ensure_runtime_dependencies()
    started = time.time()
    root, rows, groups = find_data(args.data_root, args.seed)
    development = rows
    old_counts = Counter(r["old_split"] for r in rows)
    cross_split_groups = sum(
        len({rows[i]["old_split"] for i in members}) > 1 for members in groups.values())
    print("=" * 78)
    print("SESSION 22 - YOLOv9e-seg grouped five-fold OOF on Image_Annotation_rebuilt")
    print("=" * 78)
    print(f"Dataset: {root}\nImages: {len(development)}; supplied folders: {dict(old_counts)}")
    print(f"Inferred/exact groups: {len(groups)}; groups crossing supplied splits: {cross_split_groups}")
    print("Fold sizes:", {fold: sum(int(r["cv_fold"]) == fold for r in development) for fold in range(5)})
    if args.audit_only:
        print("AUDIT PASSED: every image occurs once and no inferred/exact group crosses OOF folds.")
        return 0

    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Session 22 requires Kaggle GPU T4 x2; fewer than two CUDA GPUs detected.")
    try:
        import ultralytics
        from ultralytics import YOLO
        import pycocotools
    except ImportError as error:
        raise RuntimeError("Session 22 dependency bootstrap completed but imports still failed; restart Kaggle and run all cells.") from error

    stamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
    if args.work_dir is None:
        args.work_dir = f"/kaggle/working/session22_yolov9e_seg_original_seed{args.seed}_{stamp}"
    work = unique_output_directory(args.work_dir)
    args.work_dir = str(work)
    work.mkdir(parents=True, exist_ok=False)
    fold_info = make_fold_files(root, rows, work)
    write_prepared_manifest(rows, work / "grouped_fold_manifest.csv")
    gt_path = work / "oof_ground_truth.json"
    image_ids = build_coco_ground_truth(root, rows, gt_path)

    # Download and validate the official checkpoint once before concurrent workers.
    print(f"Ultralytics {ultralytics.__version__}; torch {torch.__version__}; GPUs={torch.cuda.device_count()}")
    checkpoint = work / MODEL
    loaded = YOLO(args.model)
    source_checkpoint = Path(getattr(loaded, "ckpt_path", None) or args.model)
    if not source_checkpoint.is_file():
        source_checkpoint = Path(args.model)
    if not source_checkpoint.is_file():
        raise RuntimeError(f"Could not resolve downloaded checkpoint: {args.model}")
    if source_checkpoint.resolve() != checkpoint.resolve():
        shutil.copy2(source_checkpoint, checkpoint)
    if getattr(loaded, "task", None) != "segment":
        raise RuntimeError(f"{args.model} is not a segmentation checkpoint")
    del loaded
    # Ultralytics performs an AMP compatibility probe with yolo11n.pt inside
    # every trainer. Cache that small checkpoint once before the two workers
    # start, preventing simultaneous downloads from racing/corrupting it.
    amp_probe = YOLO("yolo11n.pt")
    del amp_probe
    args.model = str(checkpoint)

    config = vars(args).copy()
    config.update({"build": BUILD, "classes": CLASSES, "dataset": str(root),
                   "grouped_fold_manifest_sha256": sha256(work / "grouped_fold_manifest.csv"),
                   "images": len(development), "old_split_counts": dict(old_counts),
                   "inferred_exact_groups": len(groups), "groups_crossing_supplied_splits": cross_split_groups,
                   "dataset_scope": "all supplied train/valid/test folders pooled for grouped OOF",
                   "separate_test_set": False, "patient_grouping_verified": False,
                   "ultralytics": ultralytics.__version__,
                   "torch": torch.__version__, "gpu_names": [torch.cuda.get_device_name(i) for i in range(2)]})
    source_script = resolve_runtime_script()
    script = work / "SESSION_22_runtime.py"
    if source_script.resolve() != script.resolve():
        shutil.copy2(source_script, script)
    config["runtime_sha256"] = sha256(script)
    write_json(work / "run_config.json", config)

    pending = list(range(5))
    failures = {}
    retry_counts = Counter()
    global_deadline = started + args.session_hours * 3600
    while pending:
        remaining_seconds = global_deadline - time.time() - args.reserve_minutes * 60
        waves = math.ceil(len(pending) / 2)
        if remaining_seconds <= 10 * 60:
            failures.update({fold: "insufficient global time budget" for fold in pending})
            break
        fold_hours = min(args.max_fold_hours, remaining_seconds / waves / 3600)
        wave = pending[:2]
        pending = pending[2:]
        wave_batch = args.batch
        wave_workers = args.workers
        processes = []
        for gpu, fold in enumerate(wave):
            command = [sys.executable, "-u", str(script), "--data-root", str(root),
                       "--work-dir", str(work), "--model", str(checkpoint),
                       "--worker-fold", str(fold), "--worker-gpu", str(gpu),
                       "--worker-hours", str(fold_hours), "--epochs", str(3 if args.quick else args.epochs),
                       "--patience", str(2 if args.quick else args.patience), "--imgsz", str(args.imgsz),
                       "--batch", str(args.batch), "--predict-batch", str(args.predict_batch),
                       "--workers", str(args.workers), "--seed", str(args.seed),
                       "--lr0", str(args.lr0), "--lrf", str(args.lrf),
                       "--weight-decay", str(args.weight_decay), "--conf", str(args.conf),
                       "--nms-iou", str(args.nms_iou), "--max-det", str(args.max_det)]
            log = (work / f"fold_{fold}/console.log")
            log.parent.mkdir(parents=True, exist_ok=True)
            handle = log.open("w")
            environment = os.environ.copy()
            environment.update({
                "CUDA_VISIBLE_DEVICES": str(gpu),
                "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                "WANDB_DISABLED": "true",
                "TOKENIZERS_PARALLELISM": "false",
                "PYDEVD_DISABLE_FILE_VALIDATION": "1",
                "OMP_NUM_THREADS": "2",
                "MKL_NUM_THREADS": "2",
            })
            process = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT,
                                       env=environment)
            processes.append((fold, process, handle, log))
            print(f"Started fold {fold} on GPU {gpu}; maximum {fold_hours:.2f} h; log={log}",
                  flush=True)
        last_heartbeat = time.time()
        while any(process.poll() is None for _, process, _, _ in processes):
            time.sleep(15)
            if time.time() - last_heartbeat >= 300:
                statuses = []
                for fold, process, _, log in processes:
                    last_line = "waiting for worker output"
                    if log.is_file():
                        lines = log.read_text(errors="replace").splitlines()
                        if lines:
                            last_line = lines[-1][-240:]
                    statuses.append(f"fold {fold}: {'running' if process.poll() is None else 'finished'}; {last_line}")
                print("Session 22 heartbeat | " + " | ".join(statuses), flush=True)
                last_heartbeat = time.time()
            if time.time() >= global_deadline - args.reserve_minutes * 60:
                for _, process, _, _ in processes:
                    if process.poll() is None:
                        process.terminate()
                break
        for fold, process, handle, log in processes:
            try:
                code = process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.kill(); code = process.wait()
            handle.close()
            if code != 0 or not (work / f"fold_{fold}/result.json").is_file():
                log_text = log.read_text(errors="replace") if log.is_file() else "(worker log missing)"
                tail = "\n".join(log_text.splitlines()[-120:])
                print(f"\n{'='*78}\nFOLD {fold} FAILED — worker log tail\n{'='*78}\n{tail}\n", flush=True)
                lower = log_text.lower()
                recoverable_oom = ("out of memory" in lower or "cuda error: out of memory" in lower)
                recoverable_loader = ("bus error" in lower or "dataloader worker" in lower)
                if retry_counts[fold] < 1 and recoverable_oom and wave_batch > 1:
                    retry_counts[fold] += 1
                    args.batch = min(args.batch, max(1, wave_batch // 2))
                    pending.insert(0, fold)
                    print(f"Retrying fold {fold} once with batch={args.batch}.", flush=True)
                    continue
                if retry_counts[fold] < 1 and recoverable_loader and wave_workers > 0:
                    retry_counts[fold] += 1
                    args.workers = 0
                    pending.insert(0, fold)
                    print(f"Retrying fold {fold} once with workers=0.", flush=True)
                    continue
                failures[fold] = f"exit status {code}; log={log}; tail={tail[-4000:]}"
            else:
                print(f"Fold {fold} complete")
        if failures:
            # A configuration/runtime failure usually affects every fold. Stop
            # immediately so Kaggle shows the first actionable traceback and
            # does not waste time launching the remaining folds.
            for fold in pending:
                failures.setdefault(fold, "not started after earlier fold failure")
            pending.clear()

    fold_results, pooled = [], []
    expected_ids = {r["image_id"] for r in development}
    observed_ids = []
    for fold in range(5):
        result_path = work / f"fold_{fold}/result.json"
        pred_path = work / f"fold_{fold}/oof_predictions.json"
        if not result_path.is_file() or not pred_path.is_file():
            continue
        fold_results.append(json.loads(result_path.read_text()))
        predictions = json.loads(pred_path.read_text())
        for prediction in predictions:
            string_id = prediction.pop("image_id_string")
            prediction["image_id"] = image_ids[string_id]
            pooled.append(prediction)
        valid_ids = {r["image_id"] for r in fold_info[fold]["valid"]}
        observed_ids.extend(valid_ids)
    complete = (not failures and len(fold_results) == 5 and len(observed_ids) == len(expected_ids)
                and set(observed_ids) == expected_ids)
    metrics = coco_evaluate(gt_path, pooled, work / "oof_predictions_coco.json") if complete else None
    summary = {
        "build": BUILD, "complete_five_fold_oof": complete, "separate_test_set": False,
        "dataset": str(root), "seed": args.seed, "fold_results": fold_results,
        "failed_folds": failures, "metrics": metrics,
        "metric_scope": "pooled five-fold OOF over every original folder; inferred source/exact-pixel grouped",
        "elapsed_hours": (time.time() - started) / 3600,
        "limitations": [
            "The supplied train/valid/test folders are not independent because inferred source families cross them.",
            "Source groups are filename/pixel inferred rather than patient verified.",
            "Offline augmented copies and conflicting coarse annotations remain in this experiment.",
            "Masks are coarse targets, not clinically reviewed lesion boundaries.",
            "OOF folds select checkpoints and therefore provide development evidence.",
            "One seed does not quantify seed variability.",
            "Patient identity, provenance, and permission remain unverified where manifest fields are empty.",
        ],
    }
    write_json(work / "summary.json", summary)
    archive = None
    archive_error = None
    try:
        archive = shutil.make_archive(str(work), "zip", root_dir=work)
    except (OSError, shutil.Error) as error:
        # Training and metrics remain useful if Kaggle runs short of disk while
        # packaging. Record this separately instead of misreporting model failure.
        archive_error = f"{type(error).__name__}: {error}"
        summary["archive_error"] = archive_error
        write_json(work / "summary.json", summary)
        print(f"WARNING: results are complete but ZIP creation failed: {archive_error}", flush=True)
    print("=" * 78)
    print("SESSION 22 COMPLETE" if complete else "SESSION 22 INCOMPLETE")
    if metrics:
        print(f"OOF segm AP50={metrics['segm']['AP50']:.4f}; AP50-95={metrics['segm']['AP50_95']:.4f}; "
              f"bbox AP50={metrics['bbox']['AP50']:.4f}")
    print(f"Output: {work}\nArchive: {archive or 'not created; use the output folder'}\n"
          "Separate test set: False")
    if not complete:
        raise RuntimeError(f"Five-fold OOF incomplete: {failures}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
